# Graphing SetUp

## Loading DataFrames

In [1]:
VERBOSE = False
#%run SetUp.ipynb
import pandas as pd

Home Directory = /Users/cjpar01/Work/WFE/wfey
Log Path = /Users/cjpar01/Work/WFE/wfey/logs/new_cores_and_param_run/


In [2]:
bm_output = pd.read_pickle(HOME_DIRECTORY+'/'+DF+'/'+LOGS+'_output.pkl') ## Note: Currently these 2 outputs the same, only raw
bm_output_raw = pd.read_pickle(HOME_DIRECTORY+'/'+DF+'/'+LOGS+'_output_raw.pkl')


## --- Sorting Columns --- ##
cols = bm_output_raw.columns 

energy_regex = re.compile("energy.*_input")
energy_cols = list(filter(energy_regex.match, cols))

latency_start_index = list(cols).index("SRC_ID")
latency_end_index = list(cols).index("Latency_Mean")
latency_cols = list(cols[latency_start_index:latency_end_index+1])

pdu_start_index = list(cols).index("apparentPower")
pdu_end_index = list(cols).index("powerFactor")
pdu_cols = list(cols[pdu_start_index:pdu_end_index+1])

bm_start_index = list(cols).index("epThread")
bm_cols = list(cols[bm_start_index:])

common_cols = CONSTANT_HEADERS

#print("Energy Cols: ", energy_cols)
#print("BM Cols: ", bm_cols)
print("all columns: ", cols)


## --- Flatten the arrays --- ##
bm_perf_output_flat = bm_output.explode(bm_cols, ignore_index=True)
# Makes all benchmark output numbers per sec of running
bm_perf_output_flat[bm_cols[4:]] = bm_perf_output_flat[bm_cols[4:]].div(bm_perf_output_flat.TimeRan, axis=0)

bm_latency_output_flat = bm_output.explode(latency_cols, ignore_index=True)
bm_pdu_output_flat = bm_output.explode(pdu_cols, ignore_index=True)

bm_perf_latency_output_flat = bm_perf_output_flat.explode(latency_cols, ignore_index=True)
bm_perf_pdu_output_flat = bm_perf_output_flat.explode(pdu_cols, ignore_index=True)
bm_latency_pdu_output_flat = bm_latency_output_flat.explode(pdu_cols, ignore_index=True)

bm_all_output_flat = bm_perf_latency_output_flat.explode(pdu_cols, ignore_index=True)



if (VERBOSE == True):
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, "display.precision", 3):
        display(bm_perf_latency_output_flat)
        #display(bm_output)

all columns:  Index(['KEY', 'configs', 'eventrate', 'eventprocCPUs', 'sourceCPUs',
       'workpercs', 'SRC_ID', 'WorkType', 'Latency_Min', 'Latency_Max',
       'Latency_Mean', 'apparentPower', 'current', 'realPower',
       'currentCrestFactor', 'voltage', 'energy', 'powerFactor', 'epThread',
       'ID', 'Core', 'TimeRan', 'Start Energy', 'End Energy', 'Energy Diff',
       'Total Wakeups', 'Spurious Wakeups', 'Events', 'Active Cycles',
       'Inactive Cycles', 'Cycle Diff', 'CPU Cycles', 'Instructions Retired'],
      dtype='object')


## Graph Meta Set-Up

In [3]:
sns.set_palette("Paired")
pd.options.mode.chained_assignment = None

eventsubset=[0, 50000]
sourceCPUsubset=[1, 10]
procsubset=EVENTPROCCPUS
worksubset=[0, 100]
worktypes = ['comp work', 'mem work']

bm_var_to_compare = 'Instructions Retired'
ENERGY_COLUMN='Energy Diff'


def return_df(df):
    print("Event Rate subset: ", eventsubset)
    print("Source CPU subset: ", sourceCPUsubset)
    print("Procesor CPU subset: ", procsubset)
    print("Work Perc (mem) subset: ", worksubset)
    return df[ (df['eventrate'].isin(eventsubset)) & (df['sourceCPUs'].isin(sourceCPUsubset)) & (df['eventprocCPUs'].isin(procsubset)) & (df['workpercs'].isin(worksubset))].copy()

def return_config_set(df, event, scpu):
    return ( df[ (df['eventrate'] == event) & (df['sourceCPUs'] == scpu)] )

df_used = return_df(bm_output)

Event Rate subset:  [0, 50000]
Source CPU subset:  [1, 10]
Procesor CPU subset:  [1]
Work Perc (mem) subset:  [0, 100]


## Helper Functions

In [4]:
def graph_across_source_cpus(dataset, scpu_set, eventrate_set, cmp_val1, cmp_val2, kind=None):
    df_used = return_df(dataset)
    config_set = None
    
    for sc in scpu_set:
        plt.figure()
        config_set = (df_used[(df_used['sourceCPUs'] == sc)])


        if kind == "bar":
            #fig, ax1 = plt.subplots(figsize=(10, 5))
            sns.catplot(
                data=config_set.round(2), 
                kind='bar',
                x=cmp_val1,
                y=cmp_val2,
                hue='configs',
                palette='Set2'
            )
        else:
            sns.scatterplot(
                data=config_set.round(2), 
                x=cmp_val1,
                y=cmp_val2,
                hue='configs',
                alpha=0.66,
                sizes = [200]
            )
    
        plt.title( cmp_val1 + ' vs ' + cmp_val2 + ' -- EventRate:'+str(eventrate_set)+' Source Cpus:'+str(sc))
        plt.ylabel(cmp_val2)
        plt.xlabel(cmp_val1)
        plt.tight_layout()
    

In [5]:
def bar_line_source_cpus_across_configs(dataset, scpu_set, eventrate_set, cmp_bar, cmp_line):
    df_used = return_df(dataset)
    config_set = None

    for sc in scpu_set:
        plt.figure()
        config_set = (df_used[(df_used['sourceCPUs'] == sc)])
        
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax2 = ax1.twinx()
    
        sns.barplot(
            data=config_set.round(4), 
            x='configs',
            y=cmp_bar,
            hue='eventrate',
            palette='Set2',
            errorbar=None,
            ax= ax1
        )
    
        sns.lineplot(
                data=config_set, 
                x='configs',
                y=cmp_line,
                hue='eventrate',
                palette='Set2',
                errorbar=None,
                markers=True,
                ax=ax2,
            legend=False
            )
    

        plt.title(cmp_bar + ' vs ' + cmp_line + ' -- EventRate:'+str(eventrate_set)+' Source Cpus:'+str(sc))
        ax1.set_xlabel('Config')
        ax1.set_ylabel(cmp_bar)
        ax2.set_ylabel(cmp_line)
        ax1.set_ylim(0,df_used[cmp_bar].max()) # use this to arg set per config
        ax2.set_ylim(0,df_used[cmp_line].max())
        plt.tight_layout()

In [6]:
def bar_line_eventrate_across_configs(dataset, scpu_set, eventrate_set, cmp_bar, cmp_line):
    df_used = return_df(dataset)
    config_set = None

    for e in eventrate_set:
        plt.figure()
        config_set = (df_used[(df_used['eventrate'] == e)])
        
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax2 = ax1.twinx()
    
        sns.barplot(
            data=config_set.round(4), 
            x='configs',
            y=cmp_bar,
            hue='sourceCPUs',
            palette='Set2',
            errorbar=None,
            ax= ax1
        )
    
        sns.lineplot(
                data=config_set, 
                x='configs',
                y=cmp_line,
                hue='sourceCPUs',
                palette='Set2',
                errorbar=None,
                markers=True,
                ax=ax2,
            legend=False
            )
    

        plt.title(cmp_bar + ' vs ' + cmp_line + ' -- EventRate:'+str(e)+' Source Cpus:'+str(scpu_set))
        ax1.set_xlabel('Config')
        ax1.set_ylabel(cmp_bar)
        ax2.set_ylabel(cmp_line)
        ax1.set_ylim(0,df_used[cmp_bar].max()) # use this to arg set per config
        ax2.set_ylim(0,df_used[cmp_line].max())
        plt.tight_layout()

In [7]:
def bar_bar_across_configs(dataset, scpu_set, eventrate_set, cmp_bar1, cmp_bar2):
    df_used = return_df(dataset)
    config_set = None

    for e in eventrate_set:
        for sc in scpu_set:
            plt.figure()
            config_set = return_config_set(df_used, e, sc)
            
            fig, ax1 = plt.subplots(figsize=(10, 5))

            key = str(e)+"_"+str(sc)
            try:
                ax = sns.barplot(
                    data=config_set.round(2), 
                    x='configs',
                    y=cmp_bar1,
                    errorbar=None
                )
                width_scale = 0.45
                for bar in ax.containers[0]:
                    bar.set_width(bar.get_width() * width_scale)
                ax.bar_label(ax.containers[0], fmt='%.3f',label_type='center')
    
                
                ax2 = ax.twinx()
                ax2 =sns.barplot(
                    data=config_set.round(2), 
                    x='configs',
                    y=cmp_bar2,
                    errorbar=None,
                    hatch='xx',
                    ax=ax2
                )
                for bar in ax2.containers[0]:
                    x = bar.get_x()
                    w = bar.get_width()
                    bar.set_x(x + w * (1- width_scale))
                    bar.set_width(w * width_scale)
                ax2.bar_label(ax2.containers[0], fmt='%.3E', label_type='center')
            except Exception as exc:
                print(f"{key}: {exc}")
                continue
        
    
            plt.title(cmp_bar1 + ' vs ' + cmp_bar2 + ' -- EventRate:'+str(e)+' Source Cpus:'+str(sc))
            ax1.set_xlabel('Config')
            ax1.set_ylabel(cmp_bar1)
            ax1.set_ylim(0,df_used[cmp_bar1].max()) # use this to arg set per config
            ax2.set_ylabel(cmp_bar2)
            ax2.set_ylim(0,df_used[cmp_bar2].max()) # use this to arg set per config
            plt.tight_layout()

In [8]:
def line_line_source_cpus_across_configs(dataset, scpu_set, eventrate_set, cmp_bar1, cmp_bar2):
    df_used = return_df(dataset)
    config_set = None

    for sc in scpu_set:
        plt.figure()
        config_set = (df_used[(df_used['sourceCPUs'] == sc)])
        
        fig, ax1 = plt.subplots(figsize=(10, 5))

        key = str(sc)
        try:
            ax = sns.lineplot(
                data=config_set.round(2), 
                x='configs',
                y=cmp_bar1,
                errorbar=None,
                markers=True,
            )
            width_scale = 0.45

            
            ax2 = ax.twinx()
            ax2 =sns.lineplot(
                data=config_set.round(2), 
                x='configs',
                y=cmp_bar2,
                errorbar=None,
                markers=True,
                ax=ax2,
                linestyle='--',
            )
        except Exception as exc:
            print(f"{key}: {exc.__class__.__name__}")
            continue
    

        plt.title(cmp_bar1 + ' vs ' + cmp_bar2 + ' -- EventRate: '+ str(eventrate_set) +' Source Cpus:'+str(sc))
        ax1.set_xlabel('Config')
        ax1.set_ylabel(cmp_bar1)
        ax2.set_ylabel(cmp_bar2)
        ax1.set_ylim(0,df_used[cmp_bar1].max()) # use this to arg set per config
        ax2.set_ylim(0,)
        plt.tight_layout()